# LIBRARY

In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
)
from sktime.transformations.series.adapt import TabularToSeriesAdaptor
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

from sktime.classification.shapelet_based import ShapeletTransformClassifier
from sktime.transformations.panel.shapelet_transform import RandomShapeletTransform
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sktime.classification.distance_based import KNeighborsTimeSeriesClassifier
from scikitplot.metrics import plot_roc


import numpy as np
import pandas as pd
import gzip 
import pickle 
from collections import Counter
import matplotlib.pyplot as plt
import gzip
import pickle
import sys
import numpy as np
import warnings
import pandas as pd
from scipy.stats import levene
from statsmodels.tsa.stattools import adfuller, kpss
from sklearn.preprocessing import RobustScaler
from statsmodels.tsa.seasonal import STL
from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter("ignore", InterpolationWarning)
warnings.simplefilter("ignore", UserWarning)


# DATAFRAMES

In [27]:
with gzip.open("../1.DATASET/CMI_timeseries_personalized.pkl.gz", "rb") as f:
	 CMI_timeseries_personalized = pickle.load(f)

In [28]:
# Check how many subjects/time series
print(f"Number of time series: {len(CMI_timeseries_personalized)}")

Number of time series: 4307


## Cleaning

In [29]:
df = CMI_timeseries_personalized[0].copy()
df.columns

Index(['X', 'Y', 'Z', 'enmo', 'anglez', 'non-wear_flag', 'light',
       'battery_voltage', 'weekday', 'quarter', 'relative_date_PCIAT', 'id',
       'sii_binary'],
      dtype='object')

In [30]:
cols_to_drop = ["battery_voltage", "timestamp", 'quarter', 'relative_date_PCIAT']
data_clean = [df.drop(columns=cols_to_drop, errors="ignore") for df in CMI_timeseries_personalized]

In [31]:

print(data_clean[0].columns.tolist())
all_cols = [set(df.columns) for df in data_clean]
print("All subjects have identical columns:", len(set(map(frozenset, all_cols))) == 1)


['X', 'Y', 'Z', 'enmo', 'anglez', 'non-wear_flag', 'light', 'weekday', 'id', 'sii_binary']
All subjects have identical columns: True


# NORMALIZATION

In [32]:
# Step 1: split subject IDs into train/test BEFORE scaling
all_ids    = [df["id"].iloc[0] for df in data_clean]
y_labels   = [df["sii_binary"].iloc[0] for df in data_clean]

# Filter out subjects with missing labels
valid_idx  = [i for i, label in enumerate(y_labels) if not pd.isna(label)]
valid_ids  = [all_ids[i] for i in valid_idx]
valid_labels = [y_labels[i] for i in valid_idx]

train_ids, test_ids = train_test_split(
    valid_ids,
    test_size=0.2,
    random_state=42,
    stratify=valid_labels
)

train_ids = set(train_ids)
test_ids  = set(test_ids)

# Step 2: separate the DataFrames
train_dfs = [df for df in data_clean if df["id"].iloc[0] in train_ids]
test_dfs  = [df for df in data_clean if df["id"].iloc[0] in test_ids]

In [33]:
global_scalers = {}
signals = [
    "enmo",
    "anglez",
    "light",
    "X",
    "Y",
    "Z"
]

for signal in signals:
    all_values = []

    for df in train_dfs:          # ← TRAIN ONLY
        if signal not in df.columns:
            continue
        y = df[signal].values
        if np.std(y) < 0.0001 or np.ptp(y) == 0:
            continue
        all_values.append(y)

    if not all_values:
        continue

    population = np.concatenate(all_values).reshape(-1, 1)
    q25, q75   = np.percentile(population, [25, 75])
    iqr        = q75 - q25
    signal_std = np.std(population)

    if iqr > 1e-4:
        scaler = RobustScaler()
        scaler.fit(population)
        global_scalers[signal] = ("robust", scaler)
        print(f"  Signal '{signal:<18}' → RobustScaler  (IQR={iqr:.4f})")
    elif signal_std > 1e-6:
        global_scalers[signal] = ("std_fallback", np.median(population), signal_std)
        print(f"  Signal '{signal:<18}' → Std Fallback  (IQR too small)")
    else:
        global_scalers[signal] = ("zero", None)
        print(f"  Signal '{signal:<18}' → Zeroed")



  Signal 'enmo              ' → RobustScaler  (IQR=0.0455)
  Signal 'anglez            ' → RobustScaler  (IQR=5.6422)
  Signal 'light             ' → RobustScaler  (IQR=1.8480)
  Signal 'X                 ' → RobustScaler  (IQR=0.8345)
  Signal 'Y                 ' → RobustScaler  (IQR=0.4162)
  Signal 'Z                 ' → RobustScaler  (IQR=0.4316)


In [34]:
def apply_global_scalers(df_list, global_scalers, signals):
    scaled_list = []
    for df in df_list:
        target_df  = df.copy()
        for signal, scaler_info in global_scalers.items():
            if signal not in target_df.columns:
                continue
            y    = target_df[signal].values.copy()
            kind = scaler_info[0]
            if kind == "robust":
                _, scaler = scaler_info
                y_scaled  = scaler.transform(y.reshape(-1, 1)).flatten()
            elif kind == "std_fallback":
                _, median, std = scaler_info
                y_scaled  = (y - median) / std
            else:
                y_scaled  = np.zeros_like(y)
            target_df[signal] = y_scaled
        scaled_list.append(target_df)
    return scaled_list


In [35]:
train_scaled = apply_global_scalers(train_dfs, global_scalers, signals)
test_scaled  = apply_global_scalers(test_dfs,  global_scalers, signals)

print(f"\nTrain subjects: {len(train_scaled)} | Test subjects: {len(test_scaled)}")


Train subjects: 4288 | Test subjects: 4109


# Splitting and Normalizing

In [ ]:
FEATURE_SIGNALS = ["X", "Y", "Z", "enmo", "anglez", "non-wear_flag", "light", "weekday"]
TARGET_SIGNAL   = "sii_binary"

X_list = []
y_list = []

for df in data_clean:
    target_value = df[TARGET_SIGNAL].iloc[0]
    
    if pd.isna(target_value):
        continue
    
    row = {signal: pd.Series(df[signal].values) for signal in FEATURE_SIGNALS}
    X_list.append(row)
    y_list.append(int(target_value))


X = pd.DataFrame(X_list)   # shape: (n_subjects, n_signals)
y = np.array(y_list)        # shape: (n_subjects,

In [14]:
X = pd.DataFrame(X_list)   # shape: (n_subjects, n_signals)
y = np.array(y_list)        # shape: (n_subjects,

## NORMALIZATION


In [ ]:
# ── SPLIT FIRST, THEN SCALE ───────────────────────────────────────────────────

# Step 1: split subject IDs into train/test BEFORE scaling
all_ids    = [df["id"].iloc[0] for df in CMI_timeseries_personalized]
y_labels   = [df["sii_binary"].iloc[0] for df in CMI_timeseries_personalized]

# Filter out subjects with missing labels
valid_idx  = [i for i, label in enumerate(y_labels) if not pd.isna(label)]
valid_ids  = [all_ids[i] for i in valid_idx]
valid_labels = [y_labels[i] for i in valid_idx]

train_ids, test_ids = train_test_split(
    valid_ids,
    test_size=0.2,
    random_state=42,
    stratify=valid_labels
)

train_ids = set(train_ids)
test_ids  = set(test_ids)

# Step 2: separate the DataFrames
train_dfs = [df for df in CMI_timeseries_personalized if df["id"].iloc[0] in train_ids]
test_dfs  = [df for df in CMI_timeseries_personalized if df["id"].iloc[0] in test_ids]

# ── PASS 1: FIT SCALERS ON TRAIN ONLY ────────────────────────────────────────
print("Fitting global scalers on TRAIN subjects only...")

global_scalers = {}

for signal in signals:
    all_values = []

    for df in train_dfs:          # ← TRAIN ONLY
        if signal not in df.columns:
            continue
        y = df[signal].values
        if np.std(y) < 0.0001 or np.ptp(y) == 0:
            continue
        all_values.append(y)

    if not all_values:
        continue

    population = np.concatenate(all_values).reshape(-1, 1)
    q25, q75   = np.percentile(population, [25, 75])
    iqr        = q75 - q25
    signal_std = np.std(population)

    if iqr > 1e-4:
        scaler = RobustScaler()
        scaler.fit(population)
        global_scalers[signal] = ("robust", scaler)
        print(f"  Signal '{signal:<18}' → RobustScaler  (IQR={iqr:.4f})")
    elif signal_std > 1e-6:
        global_scalers[signal] = ("std_fallback", np.median(population), signal_std)
        print(f"  Signal '{signal:<18}' → Std Fallback  (IQR too small)")
    else:
        global_scalers[signal] = ("zero", None)
        print(f"  Signal '{signal:<18}' → Zeroed")

# ── PASS 2: APPLY TO TRAIN AND TEST SEPARATELY ────────────────────────────────
def apply_global_scalers(df_list, global_scalers, signals):
    scaled_list = []
    for df in df_list:
        target_df  = df.copy()
        for signal, scaler_info in global_scalers.items():
            if signal not in target_df.columns:
                continue
            y    = target_df[signal].values.copy()
            kind = scaler_info[0]
            if kind == "robust":
                _, scaler = scaler_info
                y_scaled  = scaler.transform(y.reshape(-1, 1)).flatten()
            elif kind == "std_fallback":
                _, median, std = scaler_info
                y_scaled  = (y - median) / std
            else:
                y_scaled  = np.zeros_like(y)
            target_df[signal] = y_scaled
        scaled_list.append(target_df)
    return scaled_list

train_scaled = apply_global_scalers(train_dfs, global_scalers, signals)
test_scaled  = apply_global_scalers(test_dfs,  global_scalers, signals)

print(f"\nTrain subjects: {len(train_scaled)} | Test subjects: {len(test_scaled)}")

In [17]:
X

,X,Y,Z,enmo,anglez,non-wear_flag,light,weekday
0,0 -0.323300 1 -0.286595 2 -0.27736...,0 -0.215525 1 -0.113351 2 -0.15088...,0 0.062476 1 -0.333723 2 -0.39876...,0 0.429608 1 0.668468 2 1.84915...,0 0.054326 1 -0.137165 2 -0.16204...,0 0.0 1 0.0 2 0.0 3 0.0 4 ...,0 0.543295 1 0.520021 2 -0.07375...,0 4.0 1 4.0 2 4.0 3 4.0 4 ...
1,0 1.027581 1 1.045778 2 -0.16899...,0 -0.833655 1 -0.792908 2 -0.55839...,0 0.749501 1 0.450638 2 1.06634...,0 -0.260983 1 -0.229525 2 -0.43952...,0 0.875099 1 0.715567 2 0.98420...,0 0.0 1 0.0 2 0.0 3 0.0 4 ...,0 -0.539016 1 -0.535426 2 -0.53357...,0 1.0 1 1.0 2 1.0 3 1.0 4 ...
2,0 0.982723 1 -0.411934 2 -0.55006...,0 -0.624674 1 -0.383419 2 -0.47609...,0 1.232354 1 -0.191485 2 0.19380...,0 -0.089943 1 -0.480556 2 -0.59483...,0 1.015160 1 -0.073445 2 0.23873...,0 0.0 1 0.0 2 0.0 3 0.0 4 ...,0 1.453834 1 1.446904 2 1.43584...,0 3.0 1 3.0 2 3.0 3 3.0 4 ...
3,0 0.440504 1 0.411867 2 0.56248...,0 0.192871 1 0.037578 2 0.35625...,0 2.476029 1 2.585008 2 0.59334...,0 -0.578593 1 -0.554657 2 -0.40935...,0 1.174154 1 1.185688 2 0.88062...,0 0.0 1 0.0 2 0.0 3 0.0 4 ...,0 -0.665972 1 -0.663516 2 -0.64861...,0 5.0 1 5.0 2 5.0 3 5.0 4 ...
4,0 0.901315 1 0.066167 2 0.80806...,0 0.712711 1 0.242218 2 -0.13594...,0 0.476255 1 0.294381 2 -0.77538...,0 1.806310 1 0.458635 2 -0.60912...,0 0.723037 1 0.270185 2 -0.23829...,0 0.0 1 0.0 2 0.0 3 0.0 4 ...,0 0.444682 1 0.444682 2 0.44468...,0 5.0 1 5.0 2 5.0 3 5.0 4 ...
...,...,...,...,...,...,...,...,...
4302,0 -0.088217 1 -0.088644 2 -0.08904...,0 -1.471182 1 -1.465847 2 -1.47883...,0 0.988102 1 0.988102 2 0.98810...,0 -0.654769 1 -0.655865 2 -0.65830...,0 0.958201 1 0.958201 2 0.95820...,0 1.0 1 1.0 2 1.0 3 1.0 4 ...,0 13.201604 1 13.201604 2 13.20...,0 4.0 1 4.0 2 4.0 3 4.0 4 ...
4303,0 0.331315 1 0.337395 2 0.31317...,0 0.429593 1 0.429593 2 0.42959...,0 -1.320662 1 -1.292311 2 -1.28242...,0 -0.672514 1 -0.672514 2 -0.67251...,0 -0.368380 1 -0.364881 2 -0.35917...,0 1.0 1 1.0 2 1.0 3 1.0 4 ...,0 0.251760 1 0.251760 2 0.25176...,0 4.0 1 4.0 2 4.0 3 4.0 4 ...
4304,0 0.054883 1 0.052624 2 0.05056...,0 -0.033572 1 -0.029904 2 -0.03617...,0 -1.280856 1 -1.280645 2 -1.28019...,0 -0.672865 1 -0.672956 2 -0.67200...,0 -0.372884 1 -0.373304 2 -0.37374...,0 0.0 1 0.0 2 0.0 3 0.0 4 ...,0 -1.14993 1 -1.14993 2 -1.14993 3...,0 6.0 1 6.0 2 6.0 3 6.0 4 ...
4305,0 0.298093 1 0.298093 2 0.29809...,0 1.632137 1 1.625610 2 1.62033...,0 0.878105 1 0.865783 2 0.86598...,0 -0.662520 1 -0.662520 2 -0.66252...,0 0.925431 1 0.923030 2 0.92231...,0 1.0 1 1.0 2 1.0 3 1.0 4 ...,0 -1.170398 1 -1.159962 2 -1.15391...,0 6.0 1 6.0 2 6.0 3 6.0 4 ...


In [18]:
print(f"X shape: {X.shape}")        # (4307, 8)
print(f"y distribution: {np.bincount(y)}")

X shape: (4307, 8)
y distribution: [2887 1420]


## train and test

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        
)

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")
print(f"Train class balance: {np.bincount(y_train)}")
print(f"Test class balance:  {np.bincount(y_test)}")

Train size: 3445 | Test size: 862
Train class balance: [2309 1136]
Test class balance:  [578 284]


# Training

## parameters

In [20]:
param_grid = {
    'n_neighbors': [4, 8, 16, 32, 64, 128, 256],
    'weights': ['distance', 'uniform'],
    'distance': ['euclidean'],
    'n_jobs': [-1]
}

grid_search = GridSearchCV(
    KNeighborsTimeSeriesClassifier(),
    param_grid=param_grid,
    cv=KFold(n_splits=4, shuffle=True, random_state=42),
    n_jobs=-1,
    refit=True,
    verbose=1,
)

grid_search.fit(X_train, y_train)
knn_euclidean = grid_search.best_estimator_

Fitting 4 folds for each of 14 candidates, totalling 56 fits


KeyboardInterrupt: 